# ESOL Baseline — Week 1 ## Computational Drug Discovery: ML Generalizability Project  
**Date:** [03/08/2026] 
**Author:** [Justin Miller] 
**Objective:** Load the ESOL aqueous solubility dataset, compute Morgan fingerprints, train a Random Forest regressor, and report RMSE.  
**Hypothesis:** A random forest trained on Morgan fingerprints with a random 80/20 split will achieve RMSE < 1.0 log S on the ESOL test set. This is the baseline ceiling, the easiest possible evaluation condition.  
**Why this matters:** This is the starting point of the project. All future comparisons (scaffold split, cluster split) will be measured against this random-split baseline.

In [6]:
# Standard data tools 
import numpy as np 
# numerical arrays 
import pandas as pd
# DataFrames for structured data  
# Machine learning 
from sklearn.ensemble import RandomForestRegressor 
from sklearn.metrics import mean_squared_error  
# DeepChem for dataset loading 
import deepchem as dc  
# Plotting (we won't use these much in Week 1, but good habit to import) 
import matplotlib.pyplot as plt 
import seaborn as sns  
print("All imports successful!") 
print(f"DeepChem version: {dc.__version__}")

All imports successful!
DeepChem version: 2.8.0


## Dataset: ESOL (Aqueous Solubility)  ESOL measures aqueous solubility as log S (log molar solubility). It contains ~1,100 drug-like compounds.  
**Why it's a good starting point:** It's a clean regression problem with a well-established benchmark. Solubility is a critical ADMET property — poorly soluble drugs have poor bioavailability. This dataset is available directly through DeepChem with no manual download required.  
**Source:** Delaney (2004), MoleculeNet benchmark suite.

In [8]:
# DeepChem provides built-in loaders for benchmark datasets. 
# load_delaney() is the ESOL dataset (Delaney published it in 2004). 
# We use the 'scaffold' featurizer here just to get the SMILES strings; 
# we'll compute our own fingerprints manually below.  
tasks, datasets, transformers = dc.molnet.load_delaney(featurizer='GraphConv')  
train_dataset, val_dataset, test_dataset = datasets  
print(f"Training set size: {len(train_dataset)}") 
print(f"Validation set size: {len(val_dataset)}") 
print(f"Test set size: {len(test_dataset)}") 
print(f"Tasks: {tasks}")

Training set size: 902
Validation set size: 113
Test set size: 113
Tasks: ['measured log solubility in mols per litre']


In [13]:
# Let's look at what we actually got. 
# DeepChem datasets store molecules as SMILES strings and labels as arrays.  
# Get SMILES strings from the training set 
train_smiles = train_dataset.ids 
train_labels = train_dataset.y.flatten()  
# flatten from (N,1) to (N,)  
print("First 5 training SMILES:") 
for i in range(5):
    print(f"  {train_smiles[i]}  →  logS = {train_labels[i]:.3f}")
    print(f"\nLabel stats:") 
    print(f"  Mean logS: {train_labels.mean():.3f}")
    print(f"  Std logS:  {train_labels.std():.3f}")
    print(f"  Min logS:  {train_labels.min():.3f}")
    print(f"  Max logS:  {train_labels.max():.3f}")


First 5 training SMILES:
  CC(C)=CCCC(C)=CC(=O)  →  logS = 0.390

Label stats:
  Mean logS: -0.000
  Std logS:  1.000
  Min logS:  -4.226
  Max logS:  2.152
  CCCC=C  →  logS = 0.090

Label stats:
  Mean logS: -0.000
  Std logS:  1.000
  Min logS:  -4.226
  Max logS:  2.152
  CCCCCCCCCCCCCC  →  logS = -2.464

Label stats:
  Mean logS: -0.000
  Std logS:  1.000
  Min logS:  -4.226
  Max logS:  2.152
  CC(C)Cl  →  logS = 0.705

Label stats:
  Mean logS: -0.000
  Std logS:  1.000
  Min logS:  -4.226
  Max logS:  2.152
  CCC(C)CO  →  logS = 1.160

Label stats:
  Mean logS: -0.000
  Std logS:  1.000
  Min logS:  -4.226
  Max logS:  2.152


In [26]:
from rdkit import Chem
from rdkit.Chem import AllChem

def smiles_to_morgan(smiles_list, radius=2, n_bits=2048):
    """Convert a list of SMILES strings to Morgan fingerprint array.
    
    Args:
        smiles_list: list of SMILES strings
        radius: Morgan algorithm radius (2 = ECFP4)
        n_bits: fingerprint length in bits
    
    Returns:
        numpy array of shape (n_molecules, n_bits)
    """
    fingerprints = []
    failed = 0
    
    for smi in smiles_list:
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            fingerprints.append(np.zeros(n_bits))
            failed += 1
        else:
            fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)
            fingerprints.append(np.array(fp))
    
    if failed > 0:
        print(f"Warning: {failed} molecules failed to parse.")
    
    return np.array(fingerprints)

# Compute fingerprints for all splits
print("Computing Morgan fingerprints...")
X_train = smiles_to_morgan(train_dataset.ids)
X_val   = smiles_to_morgan(val_dataset.ids)
X_test  = smiles_to_morgan(test_dataset.ids)

y_train = train_dataset.y.flatten()
y_val   = val_dataset.y.flatten()
y_test  = test_dataset.y.flatten()

print(f"Training fingerprint matrix shape: {X_train.shape}")
print(f"Each molecule is represented as {X_train.shape[1]} binary features.")
print("Done.")


Computing Morgan fingerprints...


[23:07:41] DEPRECATION WARNING: please use MorganGenerator
[23:07:41] DEPRECATION WARNING: please use MorganGenerator
[23:07:41] DEPRECATION WARNING: please use MorganGenerator
[23:07:41] DEPRECATION WARNING: please use MorganGenerator
[23:07:41] DEPRECATION WARNING: please use MorganGenerator
[23:07:41] DEPRECATION WARNING: please use MorganGenerator
[23:07:41] DEPRECATION WARNING: please use MorganGenerator
[23:07:41] DEPRECATION WARNING: please use MorganGenerator
[23:07:41] DEPRECATION WARNING: please use MorganGenerator
[23:07:41] DEPRECATION WARNING: please use MorganGenerator
[23:07:41] DEPRECATION WARNING: please use MorganGenerator
[23:07:41] DEPRECATION WARNING: please use MorganGenerator
[23:07:41] DEPRECATION WARNING: please use MorganGenerator
[23:07:41] DEPRECATION WARNING: please use MorganGenerator
[23:07:41] DEPRECATION WARNING: please use MorganGenerator
[23:07:41] DEPRECATION WARNING: please use MorganGenerator
[23:07:41] DEPRECATION WARNING: please use MorganGenerat

Training fingerprint matrix shape: (902, 2048)
Each molecule is represented as 2048 binary features.
Done.


[23:07:46] DEPRECATION WARNING: please use MorganGenerator
[23:07:46] DEPRECATION WARNING: please use MorganGenerator
[23:07:46] DEPRECATION WARNING: please use MorganGenerator
[23:07:46] DEPRECATION WARNING: please use MorganGenerator
[23:07:46] DEPRECATION WARNING: please use MorganGenerator
[23:07:46] DEPRECATION WARNING: please use MorganGenerator
[23:07:46] DEPRECATION WARNING: please use MorganGenerator
[23:07:46] DEPRECATION WARNING: please use MorganGenerator
[23:07:46] DEPRECATION WARNING: please use MorganGenerator
[23:07:46] DEPRECATION WARNING: please use MorganGenerator
[23:07:46] DEPRECATION WARNING: please use MorganGenerator
[23:07:46] DEPRECATION WARNING: please use MorganGenerator
[23:07:46] DEPRECATION WARNING: please use MorganGenerator
[23:07:46] DEPRECATION WARNING: please use MorganGenerator
[23:07:46] DEPRECATION WARNING: please use MorganGenerator
[23:07:46] DEPRECATION WARNING: please use MorganGenerator
[23:07:46] DEPRECATION WARNING: please use MorganGenerat

In [27]:
# Random Forest builds many decision trees and averages their predictions.
# It's robust, doesn't require feature scaling, and gives feature importances.
#
# n_estimators=100: build 100 trees (standard starting point)
# random_state=42: set seed for reproducibility (use 42 throughout the project)

rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1   # use all available CPU cores to speed up training
)

print("Training Random Forest...")
rf_model.fit(X_train, y_train)
print("Training complete.")

Training Random Forest...
Training complete.


In [28]:
# RMSE = Root Mean Squared Error
# Lower is better. Units are the same as the target (log S here).
# RMSE < 1.0 log S is generally considered good for ESOL.

# Generate predictions on the test set
y_pred_test  = rf_model.predict(X_test)
y_pred_train = rf_model.predict(X_train)

# Compute RMSE
rmse_test  = np.sqrt(mean_squared_error(y_test, y_pred_test))
rmse_train = np.sqrt(mean_squared_error(y_train, y_pred_train))

print("=" * 40)
print("ESOL — Random Forest Baseline")
print("Split type: Random (DeepChem default)")
print("=" * 40)
print(f"Training RMSE: {rmse_train:.3f} log S")
print(f"Test RMSE:     {rmse_test:.3f} log S")
print()

# Quick sanity check
if rmse_test < rmse_train:
    print("Note: Test RMSE < Train RMSE. This would be unusual.")
    print("Check if there is data leakage in the split.")
else:
    gap = rmse_test - rmse_train
    print(f"Generalization gap (test - train): {gap:.3f} log S")
    if gap < 0.3:
        print("Small gap: model generalizes well on random split (expected).")
    else:
        print("Larger gap: some overfitting on training data.")

ESOL — Random Forest Baseline
Split type: Random (DeepChem default)
Training RMSE: 0.226 log S
Test RMSE:     0.799 log S

Generalization gap (test - train): 0.573 log S
Larger gap: some overfitting on training data.


## Interpretation: Week 1 Baseline

**What was built:** A Random Forest regressor trained on Morgan fingerprints 
(ECFP4, radius=2, 2048 bits) to predict ESOL aqueous solubility.

**Results:** 
- Training RMSE: 0.226 log S
- Test RMSE: 0.799 log S

**Was the hypothesis confirmed?** 
Our hypothesis, a random forest trained on Morgan fingerprints with a random 80/20 split will achieve RMSE < 1.0 log S on the ESOL test set, was confirmed. As mentioned earlier, this was expected due to the ease of evaluation.

**What this means:** This is our random-split baseline, the easiest evaluation 
condition. In later weeks, we will re-run this exact model under scaffold and 
cluster splits to measure how much performance degrades as chemical space becomes 
harder to generalize across.

The gap between random-split performance and scaffold-split performance is the 
first core finding of this project.

**Next steps (Week 2–3):** Literature review, load all three datasets, begin 
exploratory data analysis.